In [455]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score


In [456]:
def sigmoid(x):
        return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
        return sigmoid(x) * (1 - sigmoid(x))

In [457]:
def network_initialize(input_feauter, n_hiddens=[3], output_feature=1):
    network = []
    network.append({'weights': np.random.uniform(-1, 1, size=(input_feauter+1, n_hiddens[0]))})
    for i in range(1, len(n_hiddens)):
        network.append({'weights': np.random.uniform(0, 1, size=(n_hiddens[i-1]+1, n_hiddens[i]))})
    network.append({'weights': np.random.uniform(0, 1, size=(n_hiddens[-1]+1, output_feature))})
    return network


# network_initialize(3, [2,2], 1)

In [458]:
def feedforward(network, X):
    net = np.dot(X, network[0]['weights'][:-1]) + network[0]['weights'][-1]
    network[0]['net'] = net
    network[0]['out'] = sigmoid(net)
    for i in range(1, len(network)):
        net = np.dot(network[i - 1]['out'], network[i]['weights'][:-1]) + network[i]['weights'][-1]
        network[i]['net'] = net
        network[i]['out'] = sigmoid(net)
    return network


# network = network_initialize(1, [2], 1)
# print(network)
# network = feedforward(network, [[2],[3]])
# print(network)

In [459]:

def backpropagation(network, X, y, learning_rate):
    # Forward pass
    network = feedforward(network, X)
    
    # Calculate error for the output layer
    error = -2*(y - network[-1]['out'])
    delta_output = error * sigmoid_derivative(network[-1]['out'])
    # Update weights for the output layer

    network[-1]['weights'][:-1] -= learning_rate * np.dot(network[-2]['out'].T, delta_output)
    network[-1]['weights'][-1]  -= learning_rate * np.sum(delta_output, axis=0)  # Update bias
    
    # Calculate error for hidden layers and update weights
    for i in range(len(network) - 2, 0, -1):
        error = np.dot(delta_output, network[i + 1]['weights'][:-1].T)
        delta_hidden = error * sigmoid_derivative(network[i]['out'])
        network[i]['weights'][:-1] -= learning_rate * np.dot(network[i - 1]['out'].T, delta_hidden)
        network[i]['weights'][-1]  -= learning_rate * np.sum(delta_hidden, axis=0)  # Update bias
        delta_output = delta_hidden
    
    # Update weights for the first hidden layer
    delta_output = delta_output
    error = np.dot(delta_output, network[1]['weights'][:-1].T)
    delta_hidden = error * sigmoid_derivative(network[0]['out'])
    network[0]['weights'][:-1] -= learning_rate * np.dot(X.T, delta_hidden)
    network[0]['weights'][-1]  -= learning_rate * np.sum(delta_hidden, axis=0)  # Update bias
    
    return network

In [460]:
def train(X, y, epochs, learning_rate, test_x, test_y):
    network = network_initialize(input_feauter=X.shape[1], n_hiddens=[3,3], output_feature=1)
    for epoch in range(epochs):
        network = backpropagation(network, X, y, learning_rate)
        # print(network)
        if epoch % 50 == 0:
            print(f"epoch {epoch} finished", end=' | acc:')
            y_pred = prediction(test_x, network)
            acc = accuracy_score(test_y, y_pred)
            print(acc, end=' | loss:')
            print(np.mean(cross_entropy_loss(test_y, y_pred)))
            if acc == 1: break
            
    return network

def prediction(X, trained_network):
    y_pred = np.round(feedforward(trained_network, X)[-1]['out'])
    return y_pred

def cross_entropy_loss(y_true, y_pred):
    # Clip y_pred to prevent log(0) errors
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    # Compute cross-entropy
    loss = - (y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return loss
        
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])
trained_network = train(X, y, 10000, .8, X, y)

epoch 0 finished | acc:0.5 | loss:17.26978799617044
epoch 50 finished | acc:0.5 | loss:17.26958809681289
epoch 100 finished | acc:0.5 | loss:17.26958809681289
epoch 150 finished | acc:0.25 | loss:25.904482094898107
epoch 200 finished | acc:0.25 | loss:25.904482094898107
epoch 250 finished | acc:0.5 | loss:17.26958809681289
epoch 300 finished | acc:0.5 | loss:17.26958809681289
epoch 350 finished | acc:0.5 | loss:17.26958809681289
epoch 400 finished | acc:0.5 | loss:17.26958809681289
epoch 450 finished | acc:0.5 | loss:17.26958809681289
epoch 500 finished | acc:0.5 | loss:17.26958809681289
epoch 550 finished | acc:0.5 | loss:17.26958809681289
epoch 600 finished | acc:0.5 | loss:17.26958809681289
epoch 650 finished | acc:0.75 | loss:8.63489399808522
epoch 700 finished | acc:0.75 | loss:8.63489399808522
epoch 750 finished | acc:0.75 | loss:8.63489399808522
epoch 800 finished | acc:0.75 | loss:8.63489399808522
epoch 850 finished | acc:0.75 | loss:8.63489399808522
epoch 900 finished | acc:0.

In [461]:
def load_data(): 
    train_loaded = np.load(f'train_data.npz')
    test_loaded = np.load(f'test_data.npz')
    
    train_x, train_y = train_loaded['x'].T, train_loaded['y'] 
    test_x, test_y = test_loaded['x'].T, test_loaded['y'] 

    # Standarize data
    train_x = train_x / 255.
    test_x = test_x / 255.

    # Shuffle data (no effect for full batch method)
    train_indices = np.random.permutation(train_x.shape[1])
    train_x = train_x[:, train_indices]
    train_y = train_y[train_indices]

    test_indices = np.random.permutation(test_x.shape[1])
    test_x = test_x[:, test_indices]
    test_y = test_y[test_indices]

    return train_x.T, train_y.reshape(-1, 1), test_x.T, test_y.reshape(-1, 1)

train_x, train_y, test_x, test_y = load_data()
trained_network = train(train_x, train_y, 1000, 0.001, test_x, test_y)

epoch 0 finished | acc:0.5 | loss:17.269787996170436
epoch 50 finished | acc:0.79 | loss:7.253217005693537


C:\Users\top\AppData\Local\Temp\ipykernel_14708\493870331.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


epoch 100 finished | acc:0.8375 | loss:5.612681098755393


C:\Users\top\AppData\Local\Temp\ipykernel_14708\493870331.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


epoch 150 finished | acc:0.895 | loss:3.626655479195793


...

C:\Users\top\AppData\Local\Temp\ipykernel_14708\493870331.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


epoch 900 finished | acc:0.975 | loss:0.8634893998085229


C:\Users\top\AppData\Local\Temp\ipykernel_14708\493870331.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


epoch 950 finished | acc:0.99 | loss:0.3453957599234097


C:\Users\top\AppData\Local\Temp\ipykernel_14708\493870331.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
